In [ ]:
pip install -q google-adk pandas matplotlib

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6Kr_5JXQSE6a-kWmNEvY_ZIhVvHPh5VjDPp_EHU6x0ldg"

In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello!"
)

print(response.text)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from google.adk.agents import Agent
from google.adk.runners import Runner

In [ ]:
sales = pd.DataFrame(
    {
        "Month": [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun"
        ],
        "Sales": [
            120,
            150,
            170,
            165,
            210,
            240
        ],
        "Profit": [
            30,
            38,
            44,
            42,
            60,
            71
        ]
    }
)

sales.to_csv("sales.csv", index=False)

In [ ]:
df = sales

In [ ]:
def load_sales_data() -> str:
    """
    Loads the sales dataset.
    """
    global df
    df = pd.read_csv("sales.csv")

    return f"Loaded {len(df)} rows."

In [ ]:
def summarize_dataset() -> str:
    """
    Returns summary statistics.
    """
    return df.describe().to_markdown()

In [ ]:
def highest_sales() -> str:
    """
    Finds the month with the highest sales.
    """

    row = df.loc[df["Sales"].idxmax()]

    return (
        f"{row['Month']} "
        f"had the highest sales "
        f"with {row['Sales']} units."
    )

In [ ]:
def average_profit() -> str:
    """
    Calculates average profit.
    """

    return str(df["Profit"].mean())

In [ ]:
def plot_sales() -> str:
    """
    Creates a line plot of sales.
    """

    plt.figure(figsize=(8,4))

    plt.plot(df["Month"], df["Sales"], marker="o")

    plt.title("Monthly Sales")

    plt.xlabel("Month")

    plt.ylabel("Sales")

    plt.grid(True)

    plt.savefig("sales.png")

    plt.close()

    return "sales.png"

In [ ]:
analyst = Agent(
    name="sales_analyst",

    model="gemini-3.6-flash",

    instruction="""
You are a senior data analyst.

Whenever the user asks about sales,
ALWAYS use the available tools.

Never invent statistics.

Explain your reasoning clearly.
""",

    tools=[
        load_sales_data,
        summarize_dataset,
        highest_sales,
        average_profit,
        plot_sales
    ]
)

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()

runner = Runner(
    app_name="sales_analyst_app",
    agent=analyst,
    session_service=session_service,
)

In [ ]:
APP_NAME = "sales_analyst_app"
USER_ID = "soham"
SESSION_ID = "session_001"

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

print(session)

In [ ]:
from google.genai import types

events = runner.run(
    user_id=USER_ID,
    session_id=SESSION_ID,
    new_message=types.Content(
        role="user",
        parts=[
            types.Part(text="Which month had the highest sales?")
        ],
    ),
)

for event in events:
    print("=" * 80)
    print(event)

In [ ]:
import inspect

print(dir(runner))

In [ ]:
import inspect

print(inspect.signature(runner.run))

In [ ]:
import gradio as gr
from google.genai import types

# Ensure the session exists once before launching
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

def chat(message, history):

    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=message)],
        ),
    )

    answer = ""
    tool_calls = []

    for event in events:

        # Capture tool calls
        if event.content:
            for part in event.content.parts:

                if getattr(part, "function_call", None):
                    tool_calls.append(part.function_call.name)

                if getattr(part, "text", None):
                    answer += part.text

        # Capture tool errors
        if getattr(event, "error_message", None):
            answer += f"\n\n❌ {event.error_message}"

    if tool_calls:
        answer += "\n\n---\n### Tools Used\n"
        for tool in tool_calls:
            answer += f"✅ {tool}\n"

    return answer


demo = gr.ChatInterface(
    fn=chat,
    title="🤖 Google ADK Sales Analyst",
    description="""
Ask questions about the sales data.

Examples:
• Which month had the highest sales?
• What is the average profit?
• Summarize the dataset.
• Plot the sales trend.
""",
)

demo.launch(debug=True)